# Etapa 3 — Identidad del dinosaurio con Ollama (notebook SageMaker `ollama`, ml.m5.xlarge)

Ollama corre en Docker dentro de esta instancia (lo levanta el lifecycle script `on-start.sh`). Ninguna solicitud sale a un proveedor externo.

In [ ]:
!docker info --format 'data-root de Docker: {{.DockerRootDir}}'
!docker ps --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'
!df -h /home/ec2-user/SageMaker | tail -1

## Modelo descargado: nombre y versión

In [ ]:
import requests, json
OLLAMA = 'http://localhost:11434'
print('Ollama', requests.get(f'{OLLAMA}/api/version').json()['version'])
for m in requests.get(f'{OLLAMA}/api/tags').json()['models']:
    d = m['details']
    print(f"{m['name']:15} digest {m['digest'][:12]}  {d['parameter_size']}  {d['quantization_level']}  {m['size']/1e9:.1f} GB")

## Prueba: el modelo recibe una instrucción y responde

In [ ]:
r = requests.post(f'{OLLAMA}/api/generate', json={'model': 'gemma4:e2b', 'prompt': 'Responde en una frase: ¿qué es un dinosaurio?', 'stream': False}).json()
print(r['response'])
print(f"{r['eval_count']} tokens en {r['total_duration']/1e9:.1f} s")

## Identidad del dinosaurio seleccionado en la etapa 2
Se le dan al modelo ejemplos de raíces reales (-saurus, -raptor, -odon, -venator…) y se pide un JSON con apariencia, hábitat, alimentación, comportamiento y rasgo distintivo. Es el mismo prompt que usa la app (`4_app/lambda/lambda_function.py`).

In [ ]:
NOMBRE = 'Limedsaurus'
PROMPT = open('/home/ec2-user/SageMaker/lab-dinos/prompt_identidad.txt').read().format(nombre=NOMBRE)
r = requests.post(f'{OLLAMA}/api/chat', json={
    'model': 'gemma4:e2b', 'stream': False, 'think': False, 'format': 'json', 'options': {'temperature': 0.8},
    'messages': [{'role': 'user', 'content': PROMPT}]}).json()
identidad = json.loads(r['message']['content'])
for k, v in identidad.items():
    print(f'{k:20} {v}')
json.dump({'nombre': NOMBRE, **identidad}, open(f'/home/ec2-user/SageMaker/lab-dinos/identidad_{NOMBRE.lower()}.json', 'w'), ensure_ascii=False, indent=1)